# Calibrate AnomalyCLIP thresholds for MVTec, VisA, and both

Set `DATASETS = ('mvtec', 'visa', 'both')` to calibrate all collections in one run. Calibration uses only each collection's normal training images—never test images or labeled anomalies—and saves a separate checkpoint-specific artifact for every mode. Cross-dataset experiments reuse the destination dataset's artifact.

In [ ]:
import subprocess
import sys
from pathlib import Path

print('===== STEP 1: CLONE/UPDATE REPOSITORIES =====')
working = Path('/kaggle/working')
repo_root = working / 'adversarial-robustness'
anomalyclip_root = working / 'AnomalyCLIP'

repo_url = 'https://github.com/Parsagh05/adversarial-robustness.git'
if repo_root.exists():
    subprocess.run(['git', '-C', str(repo_root), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', repo_url, str(repo_root)], check=True)

official_url = 'https://github.com/zqhang/AnomalyCLIP.git'
if anomalyclip_root.exists():
    subprocess.run(['git', '-C', str(anomalyclip_root), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', official_url, str(anomalyclip_root)], check=True)

experiment_root = repo_root
requirements = experiment_root / 'requirements.txt'
print('===== STEP 2: INSTALL DEPENDENCIES =====')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)], check=True)
print('Environment ready.')

In [ ]:
import json
import os
import sys
from pathlib import Path
import torch

experiment_root = Path('/kaggle/working/adversarial-robustness')
if str(experiment_root) not in sys.path:
    sys.path.insert(0, str(experiment_root))

from adversarial_harness import AttackConfig, ExperimentConfig, calibrate_thresholds

print('===== STEP 3: RESOLVE DATA AND CHECKPOINT =====')
# Generate all three. Use a one-item tuple to calibrate only one mode.
DATASETS = ('mvtec', 'visa', 'both')
MVTEC_ROOT = Path('/kaggle/input/datasets/alirezasalehy/mvtec-ad/mvtec_anomaly_detection')
VISA_ROOT = Path('/kaggle/input/datasets/alirezasalehy/visa-ad/VisA_20220922')
ANOMALYCLIP_ROOT = Path('/kaggle/working/AnomalyCLIP')
BOTH_CHECKPOINT_DIR = '9_12_4_multiscale'
CHECKPOINT_DIRS = {
    'mvtec': '9_12_4_multiscale',       # trained on VisA
    'visa': '9_12_4_multiscale_visa',  # trained on MVTec
    'both': BOTH_CHECKPOINT_DIR,
}
OUTPUT_ROOT = Path('/kaggle/working/anomalyclip_thresholds_q95')

VALID_DATASETS = {'mvtec', 'visa', 'both'}
if not DATASETS:
    raise ValueError('DATASETS cannot be empty')
unknown = sorted(set(DATASETS) - VALID_DATASETS)
if unknown:
    raise ValueError(f'Unknown DATASETS values: {unknown}')
if len(set(DATASETS)) != len(DATASETS):
    raise ValueError(f'DATASETS contains duplicates: {DATASETS}')
if set(DATASETS) & {'mvtec', 'both'} and not MVTEC_ROOT.is_dir():
    raise FileNotFoundError(f'MVTec mount not found: {MVTEC_ROOT}')
if set(DATASETS) & {'visa', 'both'} and not VISA_ROOT.is_dir():
    raise FileNotFoundError(f'VisA mount not found: {VISA_ROOT}')
checkpoint_paths = {
    dataset: ANOMALYCLIP_ROOT / 'checkpoints' / CHECKPOINT_DIRS[dataset] / 'epoch_15.pth'
    for dataset in DATASETS
}
missing_checkpoints = [path for path in checkpoint_paths.values() if not path.is_file()]
if missing_checkpoints:
    available = sorted((ANOMALYCLIP_ROOT / 'checkpoints').rglob('*.pth'))
    raise FileNotFoundError(
        f'Checkpoints not found: {missing_checkpoints}. Available: {available}'
    )
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator before calibration.')

os.environ['ANOMALYCLIP_ROOT'] = str(ANOMALYCLIP_ROOT)

attack = AttackConfig(
    image_size=518,
    scopes=('dataset',),
    directions=('normal_to_abnormal',),
    loss_modes=('global',),
)
print('===== STEP 4: CALIBRATE ON NORMAL TRAINING IMAGES ONLY =====')
generated_thresholds = {}
for dataset in DATASETS:
    checkpoint_path = checkpoint_paths[dataset]
    dataset_output = OUTPUT_ROOT / dataset
    os.environ['ANOMALYCLIP_CHECKPOINT'] = str(checkpoint_path)
    config = ExperimentConfig(
        dataset=dataset,
        mvtec_root=str(MVTEC_ROOT) if dataset in {'mvtec', 'both'} else None,
        visa_root=str(VISA_ROOT) if dataset in {'visa', 'both'} else None,
        output_root=str(dataset_output),
        anomalyclip_root=str(ANOMALYCLIP_ROOT),
        anomalyclip_checkpoint=str(checkpoint_path),
        device='cuda',
        target_model='AnomalyCLIP',
        categories=None,
        target_batch_size=2,
        compute_lpips=False,
        threshold_mode='normal_train_quantile',
        threshold_quantile=0.95,
        thresholds_path=None,
        resume=True,
        attack=attack,
    )
    threshold_path = calibrate_thresholds(config)
    generated_thresholds[dataset] = threshold_path
    payload = json.loads(threshold_path.read_text(encoding='utf-8'))
    print(f'\n[{dataset}] checkpoint={checkpoint_path}')
    print('category | threshold | count | min | mean | max | std')
    for category, record in payload['categories'].items():
        print(
            f"{category:12s} | {record['threshold']:.6f} | "
            f"{record['sample_count']:5d} | {record['score_min']:.6f} | "
            f"{record['score_mean']:.6f} | {record['score_max']:.6f} | "
            f"{record['score_std']:.6f}"
        )
    print('Threshold artifact:', threshold_path)
    print('Raw scores:', dataset_output / 'normal_train_scores.npz')

In [ ]:
import shutil
from pathlib import Path

archive = shutil.make_archive(
    '/kaggle/working/anomalyclip_thresholds_all_q95',
    'zip',
    root_dir=OUTPUT_ROOT.parent,
    base_dir=OUTPUT_ROOT.name,
)
print('Packaged calibration artifacts:', archive)